# OpenAI API 첫걸음 — 가장 단순한 호출부터

이 노트북은 **LLM 에게 말을 거는 가장 기본적인 방법**을 다룹니다.
LangChain 같은 프레임워크 없이 OpenAI 공식 라이브러리만 씁니다.
나중에 배울 모든 것이 결국 이 호출 위에 얹힙니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 라이브러리와 API 키 |
| **Step 3~5** | 첫 대화 — `messages` 를 만들어 보내고 답 받기 |
| **Step 6~9** | 역할 주기 — `system` 으로 성격을 정하고 다시 묻기 |
| **Step 10~11** | 응답 뜯어보기 — 무엇이 들어 있나 |

> **이 노트북에서 기억할 한 가지**
> LLM 호출은 `messages` 라는 **목록 하나를 보내고 답을 받는 일**이 전부입니다.
> 복잡해 보이는 에이전트도 결국 이 목록을 잘 만들어 주는 장치입니다.

## Step 1. 준비물 세 가지

| 가져오는 것 | 하는 일 |
|---|---|
| `os` | 환경변수를 읽는다 |
| `dotenv` | `.env` 파일을 환경변수로 올려 준다 |
| `OpenAI` | OpenAI 공식 클라이언트 |

### 왜 `.env` 를 쓰나

API 키를 코드에 직접 적으면 **git 에 그대로 올라갑니다.**
한 번 올라간 키는 나중에 지워도 커밋 기록에 남습니다.

그래서 키는 `.env` 파일에 두고, 그 파일은 `.gitignore` 로 제외합니다.
코드에는 "환경변수에서 읽어라"만 적습니다.

```
llm_edu/.env        <-  OPENAI_API_KEY=sk-...   (git 에 안 올라감)
```

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. API 키가 제대로 들어왔는지 확인

```python
load_dotenv(override=True)
```

`.env` 파일을 읽어 환경변수로 올립니다.
`override=True` 는 **이미 같은 이름의 환경변수가 있어도 `.env` 값으로 덮어쓰라**는 뜻입니다.
예전에 터미널에 설정해 둔 낡은 키가 섞이는 사고를 막아 줍니다.

### 앞 8자만 찍는 이유

```python
print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
```

키 전체를 출력하면 **노트북 결과창에 그대로 저장**되고, 그 노트북을 공유하는 순간 키가 샙니다.
있는지 없는지만 확인하면 되므로 앞부분만 봅니다.

> **확인을 먼저 하는 이유**: 나중에 호출이 실패했을 때
> 원인이 '키 문제'인지 '코드 문제'인지 바로 가를 수 있습니다.

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


## Step 3. `messages` — 대화의 기본 단위

LLM 에 보내는 것은 문자열이 아니라 **역할이 붙은 메시지들의 목록**입니다.

```python
[{"role": "user", "content": "안녕하세요..."}]
```

### 역할(role) 세 가지

| role | 누가 한 말인가 | 쓰임 |
|---|---|---|
| `system` | 개발자가 모델에게 주는 지시 | 성격·말투·규칙을 정한다 |
| `user` | 사용자가 한 말 | 실제 질문 |
| `assistant` | 모델이 한 말 | 이전 답변을 다시 넣어 **대화를 이어갈 때** 쓴다 |

지금은 가장 단순하게 `user` 하나만 넣습니다.
`messages` 를 마지막 줄에 그냥 적어 **어떤 모양인지 눈으로 확인**하고 갑니다.

In [4]:
message="안녕하세요, 만나뵈서 반갑습니다. 내 이름은 홍길동입니다."
messages = [{"role": "user", "content": message}]
messages

[{'role': 'user', 'content': '안녕하세요, 만나뵈서 반갑습니다. 내 이름은 홍길동입니다.'}]

## Step 4. 클라이언트 만들기

```python
openai = OpenAI()
```

괄호 안이 비어 있는데도 동작합니다.
`OpenAI()` 가 **환경변수 `OPENAI_API_KEY` 를 알아서 찾아 쓰기** 때문입니다.
Step 2 에서 `.env` 를 올려 두었으므로 따로 넘겨줄 필요가 없습니다.

> 키를 직접 넘기고 싶다면 `OpenAI(api_key=...)` 도 됩니다.
> 하지만 그러면 키가 코드에 적히므로 권하지 않습니다.

In [5]:
openai = OpenAI()

## Step 5. 첫 호출 — 보내고 받기

```python
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content
```

필요한 것은 **모델 이름과 `messages` 둘뿐**입니다.

### `choices[0]` 이 왜 붙나

응답의 `choices` 는 **목록**입니다. 기본값은 답을 하나만 받으므로 항상 `[0]` 입니다.
`n=3` 처럼 주면 같은 질문에 서로 다른 답을 3개 받을 수 있고, 그때 `choices` 가 길어집니다.

```
response
  └ choices[0]
      └ message
          └ content   <-  우리가 원하는 답변 글자
```

모델이 "홍길동님"이라고 이름을 불러 줍니다. 우리가 보낸 문장 안에 있던 정보입니다.

In [6]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'홍길동님, 반갑습니다. 저는 AI 어시스턴트입니다. 무엇을 도와드릴까요? 예를 들어 글쓰기나 번역, 학습 계획 세우기, 정보 검색 등 다양한 도움을 드릴 수 있습니다. 원하시는 주제나 목표를 알려주시면 바로 시작하겠습니다. 한국어로도, 필요하시면 영어로도 대화 가능합니다. 어떻게 시작할까요?'

## Step 6. `system` 프롬프트 — 모델에게 역할을 준다

여기서부터가 진짜입니다. 같은 질문이라도 **역할을 어떻게 주느냐에 따라 답이 완전히 달라집니다.**

```python
system_prompt = """
당신은 IT 보안 전문가입니다. ...
"""
```

`system` 은 사용자에게 보이지 않는 **무대 뒤 지시문**입니다. 보통 이런 것을 적습니다.

| 적는 것 | 예 |
|---|---|
| 역할 | "당신은 IT 보안 전문가입니다" |
| 범위 | "보안 외의 질문에는 답하지 마세요" |
| 말투 | "초보자도 이해할 수 있게 설명하세요" |
| 형식 | "항상 번호 매긴 목록으로 답하세요" |

> 뒤에서 배울 RAG·에이전트도 결국 이 `system` 에
> **무엇을 어떻게 적을 것인가**의 싸움입니다.

In [7]:
system_prompt = """
당신은 IT 보안 전문가입니다. 
시스템 보안에 대한 전문 지식을 제공하며 새롭게 발생하는 보안 문제에 대해서도 대처 방법을 제공합니다
"""

## Step 7. `user` 프롬프트 — 실제 질문

```python
user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?
"""
```

질문만 던지지 않고 **"나는 보안 책임자다"라는 상황을 함께 줬습니다.**
같은 질문도 묻는 사람이 누구냐에 따라 적절한 답이 달라지기 때문입니다.

| 질문자 | 기대하는 답 |
|---|---|
| 보안 책임자 | 공급망 공격, 제로데이 같은 **대응 전략** |
| 일반 직원 | 비밀번호 관리 같은 **생활 수칙** |

**상황을 적어 주는 것만으로 답의 수준이 달라집니다.**

In [8]:
user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?
"""

## Step 8. 두 역할을 하나의 목록으로

```python
messages = [{"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}]
```

`system` 을 먼저, `user` 를 뒤에 놓습니다. 순서가 곧 **모델이 읽는 순서**입니다.

> **주의**: 여기서 `messages` 를 **새로 덮어썼습니다.**
> Step 3 의 "홍길동" 메시지는 이제 목록에 없습니다.
>
> 그래서 모델은 **홍길동이 누구였는지 기억하지 못합니다.**
> LLM 호출은 매번 완전히 독립적이고, 우리가 보낸 `messages` 가 전부입니다.
> 대화를 이어가려면 지난 대화를 **우리가 직접 목록에 쌓아 보내야** 합니다.
> 이 이야기는 뒤의 `1_6_stateless` 노트북에서 자세히 다룹니다.

In [9]:
messages = [ {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

## Step 9. 역할을 준 상태로 호출

```python
response = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)
```

Step 5 와 **똑같은 함수**입니다. 달라진 것은 두 가지뿐입니다.

| | Step 5 | Step 9 |
|---|---|---|
| `messages` | `user` 1개 | `system` + `user` |
| `model` | `gpt-5-nano` | `gpt-4.1-mini` |

모델 이름 문자열만 바꾸면 다른 모델로 갈아탈 수 있습니다.
**코드는 그대로입니다.** 작고 싼 모델로 개발하고 필요할 때 큰 모델로 올리는 식으로 씁니다.

In [10]:
response = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages
)

## Step 10. 응답에 무엇이 들어 있나

보통은 답변 글자만 꺼내 쓰지만, 한 번쯤 **전체를 펼쳐 보는 것**이 좋습니다.

```python
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))
```

`ensure_ascii=False` 를 빼면 한글이 `\uc548\ub155` 처럼 깨져 보입니다.

### 눈여겨볼 항목

| 항목 | 뜻 |
|---|---|
| `choices[0].message.content` | 답변 본문 — 평소 쓰는 것 |
| `choices[0].finish_reason` | 왜 멈췄나. `stop` 은 정상 종료, `length` 는 **글자 제한에 걸려 잘림** |
| `usage.prompt_tokens` | 보낸 토큰 수 |
| `usage.completion_tokens` | 받은 토큰 수 |
| `model` | 실제로 응답한 모델 (요청한 이름과 다를 수 있다) |
| `id` | 이 응답의 고유 번호. 문제 생겼을 때 추적용 |

> **`usage` 가 곧 요금입니다.** 토큰 단위로 과금되므로
> 비용이 걱정될 때 가장 먼저 봐야 할 숫자입니다.
> `finish_reason` 이 `length` 로 나온다면 답이 중간에 잘린 것이니
> `max_tokens` 를 늘리거나 질문을 쪼개야 합니다.

In [11]:
import json
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

{
  "id": "chatcmpl-EY44A4FTXZo0AkzQ8N1p0A7VPIWwm",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "현재 IT 보안 분야에서 가장 대두되고 있는 주요 보안 이슈들은 다음과 같습니다:\n\n1. 랜섬웨어 공격 증가  \n   - 기업과 공공기관을 대상으로 한 랜섬웨어 공격이 지속적으로 증가하고 있습니다. 데이터 암호화 후 금전을 요구하는 공격은 업무 중단과 막대한 비용 손실을 초래합니다.  \n   - 대응 방안: 정기적인 백업, 네트워크 분리, 다단계 인증(MFA) 적용, 엔드포인트 보안 솔루션 도입이 필수적입니다.\n\n2. 공급망 공격 (Supply Chain Attacks)  \n   - 소프트웨어 및 하드웨어 공급망을 타깃으로 하여 일차적으로 보안을 우회하는 공격이 급증하고 있습니다. 최근 SolarWinds 사태가 대표적 사례입니다.  \n   - 대응 방안: 공급업체 보안 평가 강화, 소프트웨어 서명과 코드 무결성 검증 절차 도입, 제로트러스트(Zero Trust) 모델 적용이 권장됩니다.\n\n3. 클라우드 보안 위협 확대  \n   - 클라우드 인프라 사용 확산과 함께 misconfiguration(설정 오류), 권한 남용, API 취약점 등 다양한 위험이 증가하고 있습니다.  \n   - 대응 방안: 클라우드 보안 정책 및 모니터링 강화, IAM(Identity and Access Management) 엄격 운영, 자동화된 컴플라이언스 검사 도구 활용이 필요합니다.\n\n4. 피싱 및 사회공학 공격 증가  \n   - 사용자를 대상으로 한 지능형 피싱 공격이 고도화되어 기업 내부 계정 탈취 및 내부 네트워크 침투 시도로 이어지고 있습니다.  \n   - 대응 방안: 지속적인 보안 교육과 훈련, 이메일 필터링 및 분석 

## Step 11. 실무에서 쓰는 건 결국 이 한 줄

```python
print(response.choices[0].message.content)
```

Step 10 에서 본 긴 JSON 중 **우리가 실제로 쓰는 것은 이 한 줄**입니다.
나머지는 문제가 생겼을 때 들여다보는 정보입니다.

Step 5 와 달리 `print()` 로 감쌌습니다.
셀 마지막 줄에 그냥 두면 `'...\n...'` 처럼 줄바꿈이 글자로 보이지만,
`print()` 로 찍으면 **줄바꿈과 목록이 제대로 보입니다.**

In [12]:
print(response.choices[0].message.content)

현재 IT 보안 분야에서 가장 대두되고 있는 주요 보안 이슈들은 다음과 같습니다:

1. 랜섬웨어 공격 증가  
   - 기업과 공공기관을 대상으로 한 랜섬웨어 공격이 지속적으로 증가하고 있습니다. 데이터 암호화 후 금전을 요구하는 공격은 업무 중단과 막대한 비용 손실을 초래합니다.  
   - 대응 방안: 정기적인 백업, 네트워크 분리, 다단계 인증(MFA) 적용, 엔드포인트 보안 솔루션 도입이 필수적입니다.

2. 공급망 공격 (Supply Chain Attacks)  
   - 소프트웨어 및 하드웨어 공급망을 타깃으로 하여 일차적으로 보안을 우회하는 공격이 급증하고 있습니다. 최근 SolarWinds 사태가 대표적 사례입니다.  
   - 대응 방안: 공급업체 보안 평가 강화, 소프트웨어 서명과 코드 무결성 검증 절차 도입, 제로트러스트(Zero Trust) 모델 적용이 권장됩니다.

3. 클라우드 보안 위협 확대  
   - 클라우드 인프라 사용 확산과 함께 misconfiguration(설정 오류), 권한 남용, API 취약점 등 다양한 위험이 증가하고 있습니다.  
   - 대응 방안: 클라우드 보안 정책 및 모니터링 강화, IAM(Identity and Access Management) 엄격 운영, 자동화된 컴플라이언스 검사 도구 활용이 필요합니다.

4. 피싱 및 사회공학 공격 증가  
   - 사용자를 대상으로 한 지능형 피싱 공격이 고도화되어 기업 내부 계정 탈취 및 내부 네트워크 침투 시도로 이어지고 있습니다.  
   - 대응 방안: 지속적인 보안 교육과 훈련, 이메일 필터링 및 분석 시스템 도입, 다단계 인증 적용 등이 효과적입니다.

5. AI와 머신러닝 기반 공격 및 방어  
   - AI 기술의 발전으로 AI를 이용한 악성코드 탐지 우회, 딥페이크 등을 활용한 공격이 예상됩니다. 반면 AI 기반 방어 체계도 빠르게 발전하고 있습니다.  
   - 대응 방안: AI 보안 기술 연구 투자, 이상행위 탐지 시스템 도입, 최신 위협 인텔리

---

## 정리

이 노트북에서 한 일은 이게 전부입니다.

```
messages 목록을 만든다  ->  create() 로 보낸다  ->  choices[0].message.content 를 꺼낸다
```

### 꼭 기억할 것

| | 내용 |
|---|---|
| **키는 `.env` 에** | 코드에 적으면 git 에 올라간다 |
| **`messages` 가 전부** | 모델은 우리가 보낸 목록만 본다 |
| **`system` 이 성격을 정한다** | 같은 질문도 역할에 따라 답이 달라진다 |
| **호출은 매번 독립적** | 모델은 지난 대화를 기억하지 못한다 |
| **`usage` 가 요금** | 토큰 단위로 과금된다 |

### 직접 해 보기

1. `system_prompt` 를 "초등학생도 알아듣게 설명하세요"로 바꿔 보세요. 답이 어떻게 달라지나요?
2. `system_prompt` 를 아예 빼고 `user` 만 보내 보세요. 무엇이 달라지나요?
3. Step 3 의 "홍길동" 메시지와 `assistant` 답변을 `messages` 에 **같이 넣고**
   "내 이름이 뭐라고 했지?" 라고 물어 보세요. 이번엔 기억할 것입니다.

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_2_gemini` · `1_3_ollama` | 같은 일을 다른 모델로 — 코드가 얼마나 비슷한지 |
| `1_5_tokenizer` | 토큰이 정확히 무엇인가 |
| `1_6_stateless` | 모델이 기억하지 못한다는 것의 의미와 해결 |